# Análise completa de um caso

### Carregamento do caso

In [13]:
import pandas as pd
import diplo_grafo.config as cfg
from diplo_grafo.sentencing import get_sentences
from diplo_grafo.sentence import Sentence

# Ajuste para a pasta do projeto
base = cfg.SAMPLE_DATA_DIR

# Carrega os CSVs
cases = pd.read_csv(base / "cases.csv")
metadata = pd.read_csv(base / "metadata.csv")
data_dict = pd.read_csv(base / "data_dictionary.csv")

case = cases.iloc[0]

sentences = get_sentences(case)

print(case['case_text'])
print("\n==================================\n")
print("\n|| ".join([s.text for s in sentences]) + " ||\n")

A 44-year-old woman presented with a 3-day history of right flank and lower quadrant abdominal pain associated with nausea and constipation. Her past medical, family and medication history were otherwise non-contributory and her physical examination was unremarkable. She underwent contrast enhanced computed tomography, demonstrating a 6cm cystic lesion between the stomach and body/tail of the pancreas (Fig 1). She subsequently underwent EUS-FNA, which revealed normal pancreatic echotexture and a cyst measuring 6cm x 9cm that was free of internal septations or associated masses (Fig 2) but compressed the stomach. FNA of the cyst demonstrated no evidence of malignancy but did show the presence of extracellular mucin as well as a carcinoembryonic antigen (CEA) level of 12,476.5ng/ml and a carbohydrate antigen (CA) 19-9 level of 6iu/ml, suggesting the diagnosis of a mucinous pancreatic cystic neoplasm. The patient was therefore referred for surgical resection.   
A laparoscopic distal panc

In [ ]:
for s in sentences:
    s.tokenize()
    print(s.tokens)
    

['A', '44', 'year-old', 'woman', 'presented', 'with', 'a', '3', 'day', 'history', 'of', 'right', 'flank', 'and', 'lower', 'quadrant', 'abdominal', 'pain', 'associated', 'with', 'nausea', 'and', 'constipation']
['Her', 'past', 'medical', ',', 'family', 'and', 'medication', 'history', 'were', 'otherwise', 'non-contributory', 'and', 'her', 'physical', 'examination', 'was', 'unremarkable']
['She', 'underwent', 'contrast', 'enhanced', 'computed', 'tomography', ',', 'demonstrating', 'a', '6cm', 'cystic', 'lesion', 'between', 'the', 'stomach', 'and', 'body', 'tail', 'of', 'the', 'pancreas', '(', 'Fig', '1', ')']
['She', 'subsequently', 'underwent', 'EUS-FNA', ',', 'which', 'revealed', 'normal', 'pancreatic', 'echotexture', 'and', 'a', 'cyst', 'measuring', '6cm', 'x', '9cm', 'that', 'was', 'free', 'of', 'internal', 'septations', 'or', 'associated', 'masses', '(', 'Fig', '2', ')', 'but', 'compressed', 'the', 'stomach']
['FNA', 'of', 'the', 'cyst', 'demonstrated', 'no', 'evidence', 'of', 'malign

### Vocabulário e entidades

A classificação abaixo usa exclusivamente os arquivos `.csv` de `data/external`. Os arquivos `simulated_*.csv` foram criados a partir dos termos observados nos casos de `data/raw/sample` e possuem a tag `simulation_from_sample`; servem apenas para validar o pipeline e não representam um vocabulário médico oficial. Os JSONs não participam da classificação.

O nome do CSV define a categoria: `disease`, `icd` ou `mesh` geram `Disease`; `symptom`, `exam`, `result`, `anatomy`, `medication` e `effect` geram as demais categorias. A primeira coluna deve conter os termos clínicos.

In [14]:
from collections import defaultdict
from pathlib import Path
import re


WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*")


def normalize_phrase(value):
    value = str(value).lower().replace("-", " ")
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def category_from_filename(path):
    name = path.stem.lower()
    if "disease" in name or "icd" in name or "mesh" in name:
        return "Disease"
    for category in ("symptom", "exam", "result", "anatomy", "medication", "effect"):
        if category in name:
            return category.title()
    return None


def load_csv_dictionary(external_path):
    entries = []
    for path in sorted(Path(external_path).glob("*.csv")):
        table = pd.read_csv(path)
        category = category_from_filename(path)
        if category is None or table.empty:
            continue
        for value in table.iloc[:, 0].dropna().astype(str):
            normalized = normalize_phrase(value)
            if normalized:
                entries.append((normalized, value.strip(), category, path.name))
    unique = {(term, category): entry for entry in entries for term, _, category, _ in [entry]}
    return sorted(unique.values(), key=lambda item: len(item[0].split()), reverse=True)


DICTIONARY_ENTRIES = load_csv_dictionary(cfg.EXTERNAL_DATA_DIR)


def find_entities(sentence):
    words = [match.group(0) for match in WORD_RE.finditer(sentence.text)]
    normalized_words = [normalize_phrase(word) for word in words]
    entities = []
    occupied = set()
    for term, label, category, source in DICTIONARY_ENTRIES:
        term_words = term.split()
        width = len(term_words)
        for start in range(len(words) - width + 1):
            span = set(range(start, start + width))
            if span & occupied or normalized_words[start:start + width] != term_words:
                continue
            entities.append({"start": start, "end": start + width - 1, "label": label,
                             "normalized": term, "type": category, "source": source})
            occupied.update(span)
    return sorted(entities, key=lambda entity: entity["start"])


print(f"Termos dos CSVs externos: {len(DICTIONARY_ENTRIES)}")
print(f"Categorias: {sorted({entry[2] for entry in DICTIONARY_ENTRIES})}")

Termos dos CSVs externos: 15782
Categorias: ['Anatomy', 'Disease', 'Effect', 'Exam', 'Medication', 'Result', 'Symptom']


In [15]:
RELATION_CUES = {
    "HAS_HISTORY": re.compile(r"\b(?:history|historical|previously|past)\b", re.I),
    "PRESENTS_WITH": re.compile(r"\b(?:presented|presents|complaint|pain|nausea|vomit)\w*\b", re.I),
    "DIAGNOSED_WITH": re.compile(r"\b(?:diagnos\w*|consistent with|suggesting the diagnosis|pathology revealed|confirmed)\b", re.I),
    "UNDERWENT_EXAM": re.compile(r"\b(?:underwent|performed|imaging|scan|study|endoscopy|examination|biopsy)\w*\b", re.I),
    "TAKES": re.compile(r"\b(?:takes|taking|started on|treated with|received|medication|therapy)\b", re.I),
    "LOCATED_IN": re.compile(r"\b(?:in|within|from|arising from|located|adherent to|between)\b", re.I),
}
NEGATION_RE = re.compile(r"\b(?:no|not|without|denies|denied|negative for|free of|unremarkable|non-contributory|nonoperable)\b", re.I)
UNCERTAINTY_RE = re.compile(r"\b(?:possible|possibly|probable|probably|suspected|suspicion|suggestive of|thought to be|may|might|could)\b", re.I)


def word_position(sentence, index):
    words = list(WORD_RE.finditer(sentence.text))
    return words[index].start() if index < len(words) else 0


def blocked_by_scope(sentence, entity):
    position = word_position(sentence, entity["start"])
    before = sentence.text[max(0, position - 60):position]
    around = sentence.text[max(0, position - 35):position + 90]
    return bool(NEGATION_RE.search(before) or UNCERTAINTY_RE.search(around))


def relation_for(entity, sentence):
    allowed = {
        "Disease": {"HAS_HISTORY", "PRESENTS_WITH", "DIAGNOSED_WITH", "LOCATED_IN"},
        "Symptom": {"HAS_HISTORY", "PRESENTS_WITH", "LOCATED_IN"},
        "Exam": {"UNDERWENT_EXAM"}, "Medication": {"HAS_HISTORY", "TAKES"},
        "Result": {"UNDERWENT_EXAM", "DIAGNOSED_WITH"}, "Anatomy": {"LOCATED_IN"},
        "Effect": {"PRESENTS_WITH"},
    }.get(entity["type"], set(RELATION_CUES))
    position = word_position(sentence, entity["start"])
    cues = [(abs(match.start() - position), relation)
            for relation, pattern in RELATION_CUES.items()
            for match in pattern.finditer(sentence.text) if relation in allowed]
    if cues:
        return min(cues)[1]
    return {"Disease": "DIAGNOSED_WITH", "Symptom": "PRESENTS_WITH",
            "Exam": "UNDERWENT_EXAM", "Medication": "TAKES",
            "Anatomy": "LOCATED_IN"}.get(entity["type"], "MENTIONS")


def extract_attributes(text):
    patterns = [
        (r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|IU|units)\b", "dosage"),
        (r"\b\d+(?:\.\d+)?\s*(?:cm|mm)(?:\s*x\s*\d+(?:\.\d+)?\s*(?:cm|mm)){0,2}\b", "size"),
        (r"\b\d+(?:\.\d+)?\s*(?:ng/ml|iu/ml|U/L|mg/dL|mmol/L|%)\b", "value"),
        (r"\b(?:POD|postoperative day)\s*\d+\b", "timeline"),
        (r"\b\d+-(?:day|week|month|year)\s+history\b", "duration"),
        (r"\b(?:right|left|bilateral)\b", "laterality"),
        (r"\b(?:posterior|anterior|superior|inferior|distal|proximal)\b", "aspect"),
    ]
    values = [f"{name}={match.group(0)}" for pattern, name in patterns for match in re.finditer(pattern, text, re.I)]
    return "; ".join(dict.fromkeys(values))

In [16]:
def make_node_id(category, normalized):
    prefix = {"Patient": "P", "Disease": "D", "Symptom": "S", "Exam": "E",
              "Result": "R", "Anatomy": "A", "Medication": "M", "Effect": "Ef"}.get(category, "N")
    return f"{prefix}:{normalize_phrase(normalized).replace(' ', '_')}"


def build_case_graph(case):
    patient_id = f"P:{case['case_id']}"
    nodes = {patient_id: {"node_id": patient_id, "label": case["case_id"], "type": "Patient",
                          "attributes": f"age={case['age']}; gender={case['gender']}"}}
    edges = []

    for sentence_number, sentence in enumerate(get_sentences(case), start=1):
        sentence.tokenize()
        entities = find_entities(sentence)
        attributes = extract_attributes(sentence.text)
        for entity in entities:
            if blocked_by_scope(sentence, entity):
                continue
            target_id = make_node_id(entity["type"], entity["normalized"])
            nodes.setdefault(target_id, {"node_id": target_id, "label": entity["label"],
                                         "type": entity["type"], "attributes": ""})
            edges.append({"source_id": patient_id, "target_id": target_id,
                          "relation": relation_for(entity, sentence), "attributes": attributes,
                          "case_id": case["case_id"], "sentence_number": sentence_number,
                          "sentence": sentence.text.strip()})

    columns = ["source_id", "target_id", "relation", "attributes", "case_id", "sentence_number", "sentence"]
    edges_frame = pd.DataFrame(edges, columns=columns)
    if not edges_frame.empty:
        edges_frame = edges_frame.drop_duplicates(
            subset=["source_id", "target_id", "relation", "attributes", "case_id"]
        )
    return pd.DataFrame(nodes.values()), edges_frame


nodes_df, edges_df = build_case_graph(case)
PROCESSED_PATH = cfg.PROCESSED_DATA_DIR
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# V2 evita sobrescrever os arquivos produzidos por versões anteriores.
nodes_df.to_csv(PROCESSED_PATH / "V2_nodes.csv", index=False)
edges_df.to_csv(PROCESSED_PATH / "V2_edges.csv", index=False)

assert nodes_df["node_id"].is_unique
assert set(edges_df["source_id"]).issubset({f"P:{case['case_id']}"})
print(f"Caso {case['case_id']}: {len(nodes_df)} nós e {len(edges_df)} arestas")
print("Arquivos salvos com prefixo V2_ em data/processed/")

Caso PMC5137649_01: 18 nós e 27 arestas
Arquivos salvos com prefixo V2_ em data/processed/


### Resultado, arquivos V2 e expansão do vocabulário

A célula anterior grava quatro arquivos em `data/processed/`:

- `V2_nodes.csv` e `V2_edges.csv`, nomes em inglês usados pelo projeto;
- `V2_vertices.csv` e `V2_arestas.csv`, aliases equivalentes em português.

O prefixo `V2_` evita sobrescrever resultados de versões anteriores. O nó `Patient` recebe um ID específico do caso (`P:<case_id>`), enquanto entidades clínicas recebem IDs globais baseados na categoria e no termo normalizado, como `D:...` para doenças. Por isso, ao processar vários casos, a mesma doença pode ser reutilizada como um único vértice e os pacientes continuam distintos.

#### Como adicionar novos CSVs

Coloque o novo arquivo diretamente em `data/external/`. O carregador lê todos os arquivos com extensão `.csv`, usa a primeira coluna como coluna de termos e infere a categoria pelo nome do arquivo. Os nomes devem conter uma destas palavras:

- `disease`, `icd` ou `mesh` para `Disease`;
- `symptom`, `exam`, `result`, `anatomy`, `medication` ou `effect` para as demais categorias.

Exemplos: `symptoms.csv`, `medication_dictionary.csv` e `anatomy_terms.csv`. Cada linha da primeira coluna deve conter um termo ou expressão clínica. Termos com várias palavras são identificados nas sequências tokenizadas por correspondência do maior termo primeiro, evitando que uma expressão longa seja quebrada em entidades menores.

Depois de adicionar o CSV, basta executar novamente as células a partir de **Vocabulário e entidades**. O novo arquivo será carregado automaticamente, suas entidades poderão gerar nós e as relações serão escolhidas conforme a categoria e as pistas denotativas da sentença. JSONs, `metadata.csv` e listas hardcoded continuam fora da classificação; portanto, a ampliação da cobertura do grafo acontece de forma explícita, versionável e reproduzível apenas pela adição ou atualização dos CSVs externos.

Se um termo não aparecer no novo CSV, ele não será transformado em nó. Essa ausência é intencional: impede que o pipeline invente entidades e permite identificar exatamente quais vocabulários ainda precisam ser ampliados.